# Dataset Audition

Before choosing a dataset for further preparation, this notebook takes a quick look at each candidate's size, variable types, missing values, duplicate records, and basic distributions. These checks provide an initial picture of the data while leaving the original datasets unchanged.


In [2]:
import pandas as pd
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)


## 1. Dataset A — NSW Train Occupancy

This section performs a lightweight audit of the NSW Train Occupancy dataset.


In [39]:
train_path = Path("../data/raw/train_occupancy.csv")
train_df = pd.read_csv(train_path)


### Dataset Overview


In [41]:
print(f"Rows: {train_df.shape[0]:,}")
print(f"Columns: {train_df.shape[1]:,}")
print("Column names:")
print(train_df.columns.tolist())
display(train_df.head())


Rows: 50,250
Columns: 14
Column names:
['day', 'Actual.Stop.Station', 'Actual.Station.Arrv.Time', 'Actual.Station.Dprt.Time', 'Segment.Direction', 'Trip.Name', 'Service.Line', 'Orig..Station', 'Dest..Station', 'Leading.Set.Type', 'Node.Seq.Order', 'Actual.Station.Dprt.Time.Band', 'Occupancy Status', 'Occupancy Range']


,day,Actual.Stop.Station,Actual.Station.Arrv.Time,Actual.Station.Dprt.Time,Segment.Direction,Trip.Name,Service.Line,Orig..Station,Dest..Station,Leading.Set.Type,Node.Seq.Order,Actual.Station.Dprt.Time.Band,Occupancy Status,Occupancy Range
0,9,Miranda,2017-01-10 00:18:17,2017-01-10 00:19:04,Down,620S,Illawarra,Central,Cronulla,T,18,00:15-00:29,MANY_SEATS_AVAILABLE,Low: 0-399
1,13,Beecroft,2017-01-13 08:05:59,2017-01-13 08:06:56,Down,152C,North via Macquarie Park,Central,Hornsby,A,16,08:00-08:14,MANY_SEATS_AVAILABLE,Low: 0-399
2,12,Arncliffe,2017-01-12 07:22:51,2017-01-12 07:23:36,Down,607B,Illawarra,Central,Waterfall,T,6,07:15-07:29,MANY_SEATS_AVAILABLE,Low: 0-399
3,13,Dulwich Hill,2017-01-13 07:20:39,2017-01-13 07:21:53,Down,58-F,Bankstown,Central,Birrong,A,7,07:15-07:29,MANY_SEATS_AVAILABLE,Low: 0-399
4,11,Belmore,2017-01-11 18:38:28,2017-01-11 18:39:07,Up,33-P,Bankstown,Birrong,Central,K,7,18:30-18:44,MANY_SEATS_AVAILABLE,Low: 0-399


In [42]:
train_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 50250 entries, 0 to 50249
Data columns (total 14 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   day                            50250 non-null  int64
 1   Actual.Stop.Station            50176 non-null  str  
 2   Actual.Station.Arrv.Time       50243 non-null  str  
 3   Actual.Station.Dprt.Time       50239 non-null  str  
 4   Segment.Direction              50172 non-null  str  
 5   Trip.Name                      50250 non-null  str  
 6   Service.Line                   50191 non-null  str  
 7   Orig..Station                  50170 non-null  str  
 8   Dest..Station                  50187 non-null  str  
 9   Leading.Set.Type               50173 non-null  str  
 10  Node.Seq.Order                 50250 non-null  int64
 11  Actual.Station.Dprt.Time.Band  50250 non-null  str  
 12  Occupancy Status               50177 non-null  str  
 13  Occupancy Range            

In [43]:
display(train_df.nunique(dropna=True).rename("Unique Values").to_frame())


,Unique Values
day,8
Actual.Stop.Station,307
Actual.Station.Arrv.Time,47198
Actual.Station.Dprt.Time,43139
Segment.Direction,4
Trip.Name,3587
Service.Line,30
Orig..Station,67
Dest..Station,68
Leading.Set.Type,10


### Data Quality Check


In [44]:
missing_a = pd.DataFrame({
    "Missing Count": train_df.isna().sum(),
    "Missing Percentage": (train_df.isna().mean() * 100).round(2),
})
missing_a = missing_a[missing_a["Missing Count"] > 0].sort_values(
    "Missing Percentage",
    ascending=False,
)

if missing_a.empty:
    print("No missing values were detected.")
else:
    display(missing_a)


,Missing Count,Missing Percentage
Segment.Direction,78,0.16
Orig..Station,80,0.16
Actual.Stop.Station,74,0.15
Leading.Set.Type,77,0.15
Occupancy Status,73,0.15
Dest..Station,63,0.13
Service.Line,59,0.12
Actual.Station.Dprt.Time,11,0.02
Actual.Station.Arrv.Time,7,0.01


In [45]:
duplicate_a = int(train_df.duplicated().sum())
duplicate_percentage_a = duplicate_a / train_df.shape[0] * 100

print(f"Exact duplicate rows: {duplicate_a:,}")
print(f"Duplicate percentage: {duplicate_percentage_a:.2f}%")


Exact duplicate rows: 250
Duplicate percentage: 0.50%


In [46]:
display(train_df.dtypes.rename("Current dtype").to_frame())


,Current dtype
day,int64
Actual.Stop.Station,str
Actual.Station.Arrv.Time,str
Actual.Station.Dprt.Time,str
Segment.Direction,str
Trip.Name,str
Service.Line,str
Orig..Station,str
Dest..Station,str
Leading.Set.Type,str


### Variable Summary


In [47]:
numerical_a = train_df.select_dtypes(include="number")

if numerical_a.empty:
    print("No numerical variables were detected using the current dtypes.")
else:
    display(
        numerical_a.describe(
            percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
        ).T.round(2)
    )


,count,mean,std,min,1%,25%,50%,75%,99%,max
day,50250.0,12.81,2.03,9.0,10.0,11.0,13.0,15.0,16.0,16.0
Node.Seq.Order,50250.0,8.89,6.05,1.0,1.0,4.0,8.0,13.0,25.0,38.0


In [48]:
categorical_cols_a = train_df.select_dtypes(
    include=["object", "category"]
).columns

display(
    train_df[categorical_cols_a]
    .nunique(dropna=True)
    .rename("Unique Values")
    .to_frame()
)

low_cardinality_cols_a = [
    column
    for column in categorical_cols_a
    if train_df[column].nunique(dropna=True) <= 20
]

for column in low_cardinality_cols_a:
    print(f"Value counts for {column}:")
    display(train_df[column].value_counts(dropna=False).to_frame("Count"))


/var/folders/zp/_s3z0gds44q5ns64d1l8v7sw0000gn/T/ipykernel_60815/3169377638.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols_a = train_df.select_dtypes(


,Unique Values
Actual.Stop.Station,307
Actual.Station.Arrv.Time,47198
Actual.Station.Dprt.Time,43139
Segment.Direction,4
Trip.Name,3587
Service.Line,30
Orig..Station,67
Dest..Station,68
Leading.Set.Type,10
Actual.Station.Dprt.Time.Band,96


Value counts for Segment.Direction:


,Count
Segment.Direction,
Up,25197
Down,24817
Up,80
Down,78
NaN,78


Value counts for Leading.Set.Type:


,Count
Leading.Set.Type,
A,22914
T,9897
H,4810
M,4048
K,2678
V,2433
S,1088
C,967
J,917


Value counts for Occupancy Status:


,Count
Occupancy Status,
MANY_SEATS_AVAILABLE,48008
FEW_SEATS_AVAILABLE,1596
STANDING_ROOM_ONLY,398
MANY_SEATSAVAILABLE,165
NaN,73
few_seats_available,8
VTANDING_ROOM_ONLY,2


Value counts for Occupancy Range:


,Count
Occupancy Range,
Low: 0-399,45932
Medium: 400-799,3556
High: 800+,762


In [49]:
train_time_cols = [
    "Actual.Station.Arrv.Time",
    "Actual.Station.Dprt.Time",
    "Actual.Station.Dprt.Time.Band",
]

display(train_df[train_time_cols].head(10))
display(
    pd.DataFrame({
        "Current dtype": train_df[train_time_cols].dtypes.astype(str),
        "Missing Count": train_df[train_time_cols].isna().sum(),
        "Unique Values": train_df[train_time_cols].nunique(dropna=True),
    })
)


,Actual.Station.Arrv.Time,Actual.Station.Dprt.Time,Actual.Station.Dprt.Time.Band
0,2017-01-10 00:18:17,2017-01-10 00:19:04,00:15-00:29
1,2017-01-13 08:05:59,2017-01-13 08:06:56,08:00-08:14
2,2017-01-12 07:22:51,2017-01-12 07:23:36,07:15-07:29
3,2017-01-13 07:20:39,2017-01-13 07:21:53,07:15-07:29
4,2017-01-11 18:38:28,2017-01-11 18:39:07,18:30-18:44
5,2017-01-12 18:57:07,2017-01-12 18:57:40,18:45-18:59
6,2017-01-10 14:10:29,2017-01-10 14:11:20,14:00-14:14
7,2017-01-13 06:49:53,2017-01-13 06:51:05,06:45-06:59
8,2017-01-12 07:49:30,2017-01-12 07:50:11,07:45-07:59
9,2017-01-15 11:08:04,2017-01-15 11:08:55,11:00-11:14


,Current dtype,Missing Count,Unique Values
Actual.Station.Arrv.Time,str,7,47198
Actual.Station.Dprt.Time,str,11,43139
Actual.Station.Dprt.Time.Band,str,0,96


### Dataset A Summary


In [50]:
summary_a = {
    "Dataset": "NSW Train Occupancy",
    "Rows": train_df.shape[0],
    "Columns": train_df.shape[1],
    "Numerical Variables": len(train_df.select_dtypes(include="number").columns),
    "Categorical Variables": len(
        train_df.select_dtypes(include=["object", "category"]).columns
    ),
    "Missing Values": int(train_df.isna().sum().sum()),
    "Missing Percentage": round(train_df.isna().sum().sum() / train_df.size * 100, 2),
    "Duplicate Rows": duplicate_a,
    "Duplicate Percentage": round(duplicate_percentage_a, 2),
}

display(pd.DataFrame([summary_a]))


/var/folders/zp/_s3z0gds44q5ns64d1l8v7sw0000gn/T/ipykernel_60815/1171641619.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  train_df.select_dtypes(include=["object", "category"]).columns


,Dataset,Rows,Columns,Numerical Variables,Categorical Variables,Missing Values,Missing Percentage,Duplicate Rows,Duplicate Percentage
0,NSW Train Occupancy,50250,14,2,12,522,0.07,250,0.5


- Missingness is spread across several variables but remains very low overall.
- Exact duplicate rows are present and should be investigated during later preparation.
- Most variables are categorical or stored as objects; the arrival and departure time fields are also currently stored as objects.
- Low-cardinality value counts reveal inconsistent spacing, capitalisation, and occupancy labels as the main visible quality concern.


## 2. Dataset B — Air Travel Delay

This section performs a lightweight audit of the Air Travel Delay dataset.


In [51]:
airline_path = Path("../data/raw/airline_delay.csv")
airline_df = pd.read_csv(airline_path)


### Dataset Overview


In [52]:
print(f"Rows: {airline_df.shape[0]:,}")
print(f"Columns: {airline_df.shape[1]:,}")
print("Column names:")
print(airline_df.columns.tolist())

display(airline_df.head())


Rows: 101,000
Columns: 29
Column names:
['Year', 'Month', 'DayofMonth', 'DayOfWeek', 'DepTime', 'CRSDepTime', 'ArrTime', 'CRSArrTime', 'UniqueCarrier', 'FlightNum', 'TailNum', 'ActualElapsedTime', 'CRSElapsedTime', 'AirTime', 'ArrDelay', 'DepDelay', 'Origin', 'Dest', 'Distance', 'TaxiIn', 'TaxiOut', 'Cancelled', 'CancellationCode', 'Diverted', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']


,Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,UniqueCarrier,FlightNum,TailNum,ActualElapsedTime,CRSElapsedTime,AirTime,ArrDelay,DepDelay,Origin,Dest,Distance,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
0,2008,1,31,4,611.0,610,725.0,720,WN,548,N257WN,74.0,70,58.0,5.0,1.0,ONT,SJC,333,3.0,13.0,0,NaN,0,NaN,NaN,NaN,NaN,NaN
1,2008,1,18,5,1946.0,1935,2007.0,2010,WN,3770,N289CT,81.0,95,68.0,-3.0,11.0,TUS,LAX,451,6.0,7.0,0,NaN,0,NaN,NaN,NaN,NaN,NaN
2,2008,1,17,4,1018.0,1020,1102.0,1110,WN,12,N307SW,44.0,50,32.0,-8.0,-2.0,DAL,OKC,181,3.0,9.0,0,NaN,0,NaN,NaN,NaN,NaN,NaN
3,2008,1,10,4,2057.0,2055,2304.0,2300,WN,74,N747SA,67.0,65,48.0,4.0,2.0,MDW,SDF,271,5.0,14.0,0,NaN,0,NaN,NaN,NaN,NaN,NaN
4,2008,1,15,2,1825.0,18:25,2031.0,2050,WN,2530,N693SW,126.0,145,108.0,-19.0,0.0,SEA,LAS,866,5.0,13.0,0,NaN,0,NaN,NaN,NaN,NaN,NaN


In [53]:
airline_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 101000 entries, 0 to 100999
Data columns (total 29 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Year               101000 non-null  int64  
 1   Month              101000 non-null  int64  
 2   DayofMonth         101000 non-null  int64  
 3   DayOfWeek          101000 non-null  int64  
 4   DepTime            99873 non-null   str    
 5   CRSDepTime         101000 non-null  str    
 6   ArrTime            99719 non-null   str    
 7   CRSArrTime         101000 non-null  str    
 8   UniqueCarrier      101000 non-null  str    
 9   FlightNum          101000 non-null  int64  
 10  TailNum            96854 non-null   str    
 11  ActualElapsedTime  99690 non-null   str    
 12  CRSElapsedTime     101000 non-null  str    
 13  AirTime            99690 non-null   str    
 14  ArrDelay           97700 non-null   float64
 15  DepDelay           97850 non-null   float64
 16  Origin       

In [54]:
display(airline_df.nunique(dropna=True).rename("Unique Values").to_frame())


,Unique Values
Year,1
Month,1
DayofMonth,31
DayOfWeek,7
DepTime,2152
CRSDepTime,711
ArrTime,2271
CRSArrTime,952
UniqueCarrier,10
FlightNum,2669


### Data Quality Check


In [55]:
missing_b = pd.DataFrame({
    "Missing Count": airline_df.isna().sum(),
    "Missing Percentage": (airline_df.isna().mean() * 100).round(2),
})
missing_b = missing_b[missing_b["Missing Count"] > 0].sort_values(
    "Missing Percentage",
    ascending=False,
)

if missing_b.empty:
    print("No missing values were detected.")
else:
    display(missing_b)


,Missing Count,Missing Percentage
CancellationCode,99364,98.38
CarrierDelay,81177,80.37
WeatherDelay,81177,80.37
NASDelay,81177,80.37
SecurityDelay,81177,80.37
LateAircraftDelay,81177,80.37
TailNum,4146,4.10
ArrDelay,3300,3.27
DepDelay,3150,3.12
TaxiIn,2311,2.29


In [56]:
duplicate_b = int(airline_df.duplicated().sum())
duplicate_percentage_b = duplicate_b / airline_df.shape[0] * 100

print(f"Exact duplicate rows: {duplicate_b:,}")
print(f"Duplicate percentage: {duplicate_percentage_b:.2f}%")


Exact duplicate rows: 1,000
Duplicate percentage: 0.99%


In [57]:
display(airline_df.dtypes.rename("Current dtype").to_frame())


,Current dtype
Year,int64
Month,int64
DayofMonth,int64
DayOfWeek,int64
DepTime,str
CRSDepTime,str
ArrTime,str
CRSArrTime,str
UniqueCarrier,str
FlightNum,int64


### Variable Summary


In [58]:
numerical_b = airline_df.select_dtypes(include="number")

if numerical_b.empty:
    print("No numerical variables were detected using the current dtypes.")
else:
    display(
        numerical_b.describe(
            percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
        ).T.round(2)
    )


,count,mean,std,min,1%,25%,50%,75%,99%,max
Year,101000.0,2008.00,0.00,2008.0,2008.0,2008.0,2008.0,2008.0,2008.00,2008.0
Month,101000.0,1.00,0.00,1.0,1.0,1.0,1.0,1.0,1.00,1.0
DayofMonth,101000.0,17.09,8.36,1.0,3.0,10.0,17.0,24.0,31.00,31.0
DayOfWeek,101000.0,3.89,1.95,1.0,1.0,2.0,4.0,5.0,7.00,7.0
FlightNum,101000.0,1510.90,1185.51,1.0,11.0,502.0,1320.0,2362.0,3933.00,7676.0
ArrDelay,97700.0,5.71,30.91,-57.0,-29.0,-9.0,-2.0,10.0,145.01,500.0
DepDelay,97850.0,10.37,28.34,-44.0,-9.0,-2.0,1.0,10.0,142.00,516.0
TaxiIn,98689.0,4.77,2.98,1.0,2.0,3.0,4.0,5.0,17.00,213.0
TaxiOut,98847.0,10.93,5.99,1.0,5.0,8.0,9.0,12.0,34.00,150.0
CarrierDelay,19823.0,9.56,21.80,0.0,0.0,0.0,1.0,11.0,101.00,431.0


In [59]:
categorical_cols_b = airline_df.select_dtypes(
    include=["object", "category"]
).columns

display(
    airline_df[categorical_cols_b]
    .nunique(dropna=True)
    .rename("Unique Values")
    .to_frame()
)

low_cardinality_cols_b = [
    column
    for column in categorical_cols_b
    if airline_df[column].nunique(dropna=True) <= 20
]

for column in low_cardinality_cols_b:
    print(f"Value counts for {column}:")
    display(airline_df[column].value_counts(dropna=False).to_frame("Count"))


/var/folders/zp/_s3z0gds44q5ns64d1l8v7sw0000gn/T/ipykernel_60815/3525630551.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols_b = airline_df.select_dtypes(


,Unique Values
DepTime,2152
CRSDepTime,711
ArrTime,2271
CRSArrTime,952
UniqueCarrier,10
TailNum,622
ActualElapsedTime,581
CRSElapsedTime,273
AirTime,579
Origin,308


Value counts for UniqueCarrier:


,Count
UniqueCarrier,
WN,93112
XE,5868
W,488
wn,478
WN,473
WN,454
X,40
xe,34
XE,28


Value counts for Cancelled:


,Count
Cancelled,
0,98371
1,1632
TRUE,262
N,251
Y,249
FALSE,235


Value counts for CancellationCode:


,Count
CancellationCode,
NaN,99364
B,866
A,667
C,103


Value counts for Diverted:


,Count
Diverted,
0,99829
TRUE,270
FALSE,262
N,242
Y,239
1,158


In [60]:
flight_date_cols = ["Year", "Month", "DayofMonth", "DayOfWeek"]
flight_time_cols = ["DepTime", "CRSDepTime", "ArrTime", "CRSArrTime"]
delay_cols = ["DepDelay", "ArrDelay"]

display(airline_df[flight_date_cols].describe().T.round(2))
display(airline_df[flight_time_cols].head(10))
display(
    pd.DataFrame({
        "Current dtype": airline_df[flight_time_cols].dtypes.astype(str),
        "Missing Count": airline_df[flight_time_cols].isna().sum(),
        "Unique Values": airline_df[flight_time_cols].nunique(dropna=True),
    })
)
display(airline_df[delay_cols].describe(percentiles=[0.01, 0.99]).T.round(2))


,count,mean,std,min,25%,50%,75%,max
Year,101000.0,2008.00,0.00,2008.0,2008.0,2008.0,2008.0,2008.0
Month,101000.0,1.00,0.00,1.0,1.0,1.0,1.0,1.0
DayofMonth,101000.0,17.09,8.36,1.0,10.0,17.0,24.0,31.0
DayOfWeek,101000.0,3.89,1.95,1.0,2.0,4.0,5.0,7.0


,DepTime,CRSDepTime,ArrTime,CRSArrTime
0,611.0,610,725.0,720
1,1946.0,1935,2007.0,2010
2,1018.0,1020,1102.0,1110
3,2057.0,2055,2304.0,2300
4,1825.0,18:25,2031.0,2050
5,1710.0,1715,1802.0,1810
6,NaN,925,NaN,1025
7,2123.0,2055,2334.0,2335
8,1043.0,10.25,NaN,1215
9,1229.0,1220,1418.0,1425


,Current dtype,Missing Count,Unique Values
DepTime,str,1127,2152
CRSDepTime,str,0,711
ArrTime,str,1281,2271
CRSArrTime,str,0,952


,count,mean,std,min,1%,99%,max
DepDelay,97850.0,10.37,28.34,-44.0,-9.0,142.00,516.0
ArrDelay,97700.0,5.71,30.91,-57.0,-29.0,145.01,500.0


### Dataset B Summary


In [61]:
summary_b = {
    "Dataset": "Air Travel Delay",
    "Rows": airline_df.shape[0],
    "Columns": airline_df.shape[1],
    "Numerical Variables": len(airline_df.select_dtypes(include="number").columns),
    "Categorical Variables": len(
        airline_df.select_dtypes(include=["object", "category"]).columns
    ),
    "Missing Values": int(airline_df.isna().sum().sum()),
    "Missing Percentage": round(
        airline_df.isna().sum().sum() / airline_df.size * 100,
        2,
    ),
    "Duplicate Rows": duplicate_b,
    "Duplicate Percentage": round(duplicate_percentage_b, 2),
}

display(pd.DataFrame([summary_b]))


/var/folders/zp/_s3z0gds44q5ns64d1l8v7sw0000gn/T/ipykernel_60815/1698615670.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  airline_df.select_dtypes(include=["object", "category"]).columns


,Dataset,Rows,Columns,Numerical Variables,Categorical Variables,Missing Values,Missing Percentage,Duplicate Rows,Duplicate Percentage
0,Air Travel Delay,101000,29,14,15,525337,17.94,1000,0.99


- Missingness is concentrated in `CancellationCode` and the five delay-cause fields, with smaller gaps in actual time and delay variables.
- Exact duplicate rows are present, although their proportion is relatively small.
- Several scheduled and actual clock-time variables are stored as objects, so their formats require validation before later analysis.
- Low-cardinality value counts expose mixed encodings in carrier, cancellation, and diversion fields.


## 3. Dataset C — Hotel Booking

This section performs a lightweight audit of the Hotel Booking dataset.


In [ ]:
hotel_path =  Path("../data/raw/hotel_bookings.csv")
hotel_df = pd.read_csv(hotel_path)


### Dataset Overview


In [75]:
print(f"Rows: {hotel_df.shape[0]:,}")
print(f"Columns: {hotel_df.shape[1]:,}")
print("Column names:")
print(hotel_df.columns.tolist())
display(hotel_df.head())


Rows: 119,987
Columns: 32
Column names:
['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'reservation_status', 'reservation_status_date']


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,City Hotel,1,29,2016,February,6,5,6,14,1,0.0,0,BB,BRA,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,9.0,NaN,0,Transient,70.91,0,0,Canceled,2016-01-15
1,Resort Hotel,0,312,2017,March,10,5,2,5,2,0.0,0,HB,DEU,Groups,TA/TO,0,0,0,A,A,0,No Deposit,298.0,NaN,0,Transient-Party,56.00,0,0,Check-Out,2017-03-12
2,Resort Hotel,0,19,2016,February,9,27,0,1,2,0.0,0,BB,PRT,Complementary,TA/TO,0,0,0,A,F,0,No Deposit,5.0,NaN,0,Transient,0.00,0,0,Check-Out,2016-02-28
3,City Hotel,1,159,2017,July,27,8,1,1,2,0.0,0,SC,IRL,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,9.0,NaN,0,Transient,89.10,0,0,Canceled,2017-03-01
4,City Hotel,0,89,2016,June,27,26,2,3,2,0.0,0,BB,FRA,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,9.0,NaN,0,Transient,107.10,0,3,Check-Out,2016-07-01


In [76]:
hotel_df.info()


<class 'pandas.DataFrame'>
RangeIndex: 119987 entries, 0 to 119986
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119987 non-null  str    
 1   is_canceled                     119987 non-null  int64  
 2   lead_time                       119987 non-null  int64  
 3   arrival_date_year               119987 non-null  int64  
 4   arrival_date_month              119987 non-null  str    
 5   arrival_date_week_number        119987 non-null  int64  
 6   arrival_date_day_of_month       119987 non-null  int64  
 7   stays_in_weekend_nights         119987 non-null  int64  
 8   stays_in_week_nights            119987 non-null  int64  
 9   adults                          119987 non-null  int64  
 10  children                        119805 non-null  float64
 11  babies                          119987 non-null  int64  
 12  meal                       

In [65]:
display(hotel_df.nunique(dropna=True).rename("Unique Values").to_frame())


,Unique Values
hotel,4
is_canceled,2
lead_time,731
arrival_date_year,3
arrival_date_month,12
arrival_date_week_number,53
arrival_date_day_of_month,31
stays_in_weekend_nights,17
stays_in_week_nights,35
adults,14


### Data Quality Check


In [ ]:
missing_c = pd.DataFrame({
    "Missing Count": hotel_df.isna().sum(),
    "Missing Percentage": (hotel_df.isna().mean() * 100).round(2),
})
missing_c = missing_c[missing_c["Missing Count"] > 0].sort_values(
    "Missing Percentage",
    ascending=False,
)
if missing_c.empty:
    print("No missing values were detected.")
else:
    display(missing_c)


,Missing Count,Missing Percentage
company,113162,94.31
agent,16541,13.79
country,628,0.52
children,182,0.15
distribution_channel,164,0.14
customer_type,154,0.13
meal,138,0.12
market_segment,129,0.11


In [ ]:
duplicate_c = int(hotel_df.duplicated().sum())
duplicate_percentage_c = duplicate_c / hotel_df.shape[0] * 100
print(f"Exact duplicate rows: {duplicate_c:,}")
print(f"Duplicate percentage: {duplicate_percentage_c:.2f}%")


Exact duplicate rows: 31,328
Duplicate percentage: 26.11%


In [68]:
display(hotel_df.dtypes.rename("Current dtype").to_frame())


,Current dtype
hotel,str
is_canceled,int64
lead_time,int64
arrival_date_year,int64
arrival_date_month,str
arrival_date_week_number,int64
arrival_date_day_of_month,int64
stays_in_weekend_nights,int64
stays_in_week_nights,int64
adults,int64


### Variable Summary


In [ ]:
numerical_c = hotel_df.select_dtypes(include="number")
if numerical_c.empty:
    print("No numerical variables were detected using the current dtypes.")
else:
    display(
        numerical_c.describe(
            percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
        ).T.round(2)
    )


,count,mean,std,min,1%,25%,50%,75%,99%,max
is_canceled,119987.0,0.37,0.48,0.00,0.00,0.00,0.0,1.0,1.00,1.0
lead_time,119987.0,106.11,114.79,0.00,0.00,18.00,69.0,161.0,461.00,1636.0
arrival_date_year,119987.0,2016.16,0.71,2015.00,2015.00,2016.00,2016.0,2017.0,2017.00,2017.0
arrival_date_week_number,119987.0,27.17,13.61,1.00,2.00,16.00,28.0,38.0,53.00,53.0
arrival_date_day_of_month,119987.0,15.80,8.78,1.00,1.00,8.00,16.0,23.0,31.00,31.0
stays_in_weekend_nights,119987.0,0.93,1.00,0.00,0.00,0.00,1.0,2.0,4.00,19.0
stays_in_week_nights,119987.0,2.50,1.91,0.00,0.00,1.00,2.0,3.0,10.00,50.0
adults,119987.0,1.86,0.58,0.00,1.00,2.00,2.0,2.0,3.00,55.0
children,119805.0,0.10,0.40,0.00,0.00,0.00,0.0,0.0,2.00,10.0
babies,119987.0,0.01,0.10,0.00,0.00,0.00,0.0,0.0,0.00,10.0


In [ ]:
categorical_cols_c = hotel_df.select_dtypes(
    include=["object", "category"]
).columns
display(
    hotel_df[categorical_cols_c]
    .nunique(dropna=True)
    .rename("Unique Values")
    .to_frame()
)
low_cardinality_cols_c = [
    column
    for column in categorical_cols_c
    if hotel_df[column].nunique(dropna=True) <= 20
]
for column in low_cardinality_cols_c:
    print(f"Value counts for {column}:")
    display(hotel_df[column].value_counts(dropna=False).to_frame("Count"))


/var/folders/zp/_s3z0gds44q5ns64d1l8v7sw0000gn/T/ipykernel_60815/6559575.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols_c = hotel_df.select_dtypes(


,Unique Values
hotel,4
arrival_date_month,12
meal,10
country,177
market_segment,14
distribution_channel,9
reserved_room_type,10
assigned_room_type,12
deposit_type,3
customer_type,8


Value counts for hotel:


,Count
hotel,
City Hotel,79570
Resort Hotel,40188
CityHotel,142
ResortHotel,87


Value counts for arrival_date_month:


...

,Count
deposit_type,
No Deposit,105162
Non Refund,14662
Refundable,163


Value counts for customer_type:


,Count
customer_type,
Transient,89824
Transient-Party,25156
Contract,4083
Group,576
NaN,154
Transient,137
Transient-Party,49
contract,7
GROUP,1


Value counts for reservation_status:


,Count
reservation_status,
Check-Out,75425
Canceled,43168
No-Show,1211
CHECK-OUT,114
CANCELED,68
No-Show,1


In [ ]:
booking_date_cols = [
    "arrival_date_year",
    "arrival_date_month",
    "arrival_date_week_number",
    "arrival_date_day_of_month",
    "reservation_status_date",
]
display(hotel_df[booking_date_cols].head(10))
display(
    pd.DataFrame({
        "Current dtype": hotel_df[booking_date_cols].dtypes.astype(str),
        "Missing Count": hotel_df[booking_date_cols].isna().sum(),
        "Unique Values": hotel_df[booking_date_cols].nunique(dropna=True),
    })
)


,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,reservation_status_date
0,2016,February,6,5,2016-01-15
1,2017,March,10,5,2017-03-12
2,2016,February,9,27,2016-02-28
3,2017,July,27,8,2017-03-01
4,2016,June,27,26,2016-07-01
5,2017,August,34,24,2017-08-26
6,2016,December,49,1,2015-10-23
7,2017,August,34,24,2017-07-04
8,2015,November,48,24,2015-12-01
9,2016,August,32,1,2016-08-02


,Current dtype,Missing Count,Unique Values
arrival_date_year,int64,0,3
arrival_date_month,str,0,12
arrival_date_week_number,int64,0,53
arrival_date_day_of_month,int64,0,31
reservation_status_date,str,0,1686


### Dataset C Summary


In [72]:
summary_c = {
    "Dataset": "Hotel Booking",
    "Rows": hotel_df.shape[0],
    "Columns": hotel_df.shape[1],
    "Numerical Variables": len(hotel_df.select_dtypes(include="number").columns),
    "Categorical Variables": len(
        hotel_df.select_dtypes(include=["object", "category"]).columns
    ),
    "Missing Values": int(hotel_df.isna().sum().sum()),
    "Missing Percentage": round(hotel_df.isna().sum().sum() / hotel_df.size * 100, 2),
    "Duplicate Rows": duplicate_c,
    "Duplicate Percentage": round(duplicate_percentage_c, 2),
}

display(pd.DataFrame([summary_c]))


/var/folders/zp/_s3z0gds44q5ns64d1l8v7sw0000gn/T/ipykernel_60815/3430745798.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  hotel_df.select_dtypes(include=["object", "category"]).columns


,Dataset,Rows,Columns,Numerical Variables,Categorical Variables,Missing Values,Missing Percentage,Duplicate Rows,Duplicate Percentage
0,Hotel Booking,119987,32,20,12,131098,3.41,31328,26.11


- Missingness is concentrated in `company` and `agent`, while the remaining affected variables have much smaller gaps.
- The high proportion of exact duplicate rows is the most prominent structural concern for later investigation.
- The arrival date is distributed across several fields, while `reservation_status_date` is stored as an object.
- Numerical ranges include extreme values in fields such as `lead_time` and `adr`; these should be verified rather than treated automatically as errors.


## Cross-Dataset Summary

The following table combines only the objective summary statistics calculated after each dataset was audited separately.


In [ ]:
audit_summary = pd.DataFrame([
    summary_a,
    summary_b,
    summary_c,
])
display(audit_summary)


,Dataset,Rows,Columns,Numerical Variables,Categorical Variables,Missing Values,Missing Percentage,Duplicate Rows,Duplicate Percentage
0,NSW Train Occupancy,50250,14,2,12,522,0.07,250,0.50
1,Air Travel Delay,101000,29,14,15,525337,17.94,1000,0.99
2,Hotel Booking,119987,32,20,12,131098,3.41,31328,26.11
